In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import random
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import scipy.sparse

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_clean_dataset(base_dir: str, dataset_name: str) -> pd.DataFrame:
    csv_path = os.path.join(base_dir, "clean", f"{dataset_name}_clean.csv")
    return pd.read_csv(csv_path).reset_index(drop=True)


def load_graph_npz(base_dir: str, dataset_name: str, k: int):
    graph_path = os.path.join(base_dir, "graphs", dataset_name, f"knn_k{k}.npz")
    try:
        sparse_matrix = scipy.sparse.load_npz(graph_path)
        row, col = sparse_matrix.nonzero()
        num_nodes = sparse_matrix.shape[0]
        return row, col, num_nodes
    except Exception as e:
        data = np.load(graph_path)
        if "row" in data and "col" in data:
            row, col = data["row"], data["col"]
        elif "sources" in data and "targets" in data:
            row, col = data["sources"], data["targets"]
        else:
            raise ValueError(f"Unrecognized graph format: {list(data.keys())}. {e}")
        num_nodes = int(data["num_nodes"]) if "num_nodes" in data else int(max(row.max(), col.max()) + 1)
        return row, col, num_nodes


def encode_texts_for_dataset(df, tokenizer, transformer, device, max_length=128, batch_size=32):
    texts = df["text"].astype(str).tolist()
    all_embeddings = []
    transformer.eval()
    with torch.no_grad():
        batches = int(np.ceil(len(texts) / batch_size))
        for i in tqdm(range(batches), desc="  Encoding texts"):
            batch_texts = texts[i * batch_size:(i + 1) * batch_size]
            enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            outputs = transformer(**enc)
            cls_emb = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(cls_emb.cpu())
    return torch.cat(all_embeddings, dim=0)


def prepare_all_datasets(base_dir, model_name, max_length, batch_size, k, device, dataset_names):
    print(f"Using device: {device}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    transformer = AutoModel.from_pretrained(model_name).to(device)
    all_data = {}
    for name in dataset_names:
        print(f"\n=== Preparing dataset: {name} ===")
        df = load_clean_dataset(base_dir, name)
        print(f"  Num documents: {len(df)}")
        embeddings = encode_texts_for_dataset(df, tokenizer, transformer, device, max_length, batch_size)
        labels = torch.tensor(df["label"].values, dtype=torch.long)
        split = df["split"].values
        train_mask = torch.tensor(split == "train", dtype=torch.bool)
        val_mask = torch.tensor(split == "val", dtype=torch.bool)
        test_mask = torch.tensor(split == "test", dtype=torch.bool)
        row, col, num_nodes = load_graph_npz(base_dir, name, k)
        if num_nodes != len(df):
            num_nodes = len(df)
            row = np.clip(row, 0, num_nodes - 1)
            col = np.clip(col, 0, num_nodes - 1)
        edge_index = torch.tensor(np.vstack([row, col]), dtype=torch.long)
        all_data[name] = {"emb": embeddings, "labels": labels, "train_mask": train_mask,
                           "val_mask": val_mask, "test_mask": test_mask, "edge_index": edge_index,
                           "num_nodes": num_nodes}
    return all_data


# ---------------- Models: GCN, GAT, T-Base ----------------
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.0):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, num_nodes):
        row, col = edge_index
        device = x.device
        self_loop = torch.arange(num_nodes, device=device)
        row = torch.cat([row, self_loop], dim=0)
        col = torch.cat([col, self_loop], dim=0)
        deg = torch.zeros(num_nodes, device=device)
        deg.index_add_(0, row, torch.ones_like(row, dtype=deg.dtype, device=device))
        deg_inv_sqrt = torch.pow(deg.clamp(min=1.0), -0.5)
        x = self.linear(x)
        x = self.dropout(x)
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
        x_col = x[col] * norm.unsqueeze(-1)
        out = torch.zeros_like(x)
        out.index_add_(0, row, x_col)
        return out


class GATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.0, negative_slope=0.2):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.a_src = nn.Linear(out_dim, 1, bias=False)
        self.a_dst = nn.Linear(out_dim, 1, bias=False)
        self.leakyrelu = nn.LeakyReLU(negative_slope)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, num_nodes):
        device = x.device
        row, col = edge_index
        self_loop = torch.arange(num_nodes, device=device)
        row = torch.cat([row, self_loop], dim=0)
        col = torch.cat([col, self_loop], dim=0)
        Wh = self.W(x)
        alpha_src = self.a_src(Wh).squeeze(-1)
        alpha_dst = self.a_dst(Wh).squeeze(-1)
        e = self.leakyrelu(alpha_src[row] + alpha_dst[col])
        seg_max = torch.full((num_nodes,), float("-inf"), device=device)
        seg_max = seg_max.scatter_reduce(0, row, e, reduce="amax", include_self=True)
        exp_e = torch.exp(e - seg_max[row])
        denom = torch.zeros(num_nodes, device=device)
        denom.index_add_(0, row, exp_e)
        denom = denom.clamp(min=1e-12)
        attn = self.dropout(exp_e / denom[row])
        out = torch.zeros_like(Wh)
        out.index_add_(0, row, Wh[col] * attn.unsqueeze(-1))
        return out


class HybridGraphTransformer(nn.Module):
    def __init__(self, layer_cls, emb_dim, hidden=256, fusion_hidden=256, num_classes=2, dropout=0.3):
        super().__init__()
        self.l1 = layer_cls(emb_dim, hidden, dropout=dropout)
        self.l2 = layer_cls(hidden, hidden, dropout=dropout)
        self.fc_fuse = nn.Linear(emb_dim + hidden, fusion_hidden)
        self.fc_out = nn.Linear(fusion_hidden, num_classes)
        self.dropout = nn.Dropout(dropout)
        self.act = F.elu if layer_cls is GATLayer else F.relu

    def forward(self, x, edge_index, num_nodes):
        g = self.act(self.l1(x, edge_index, num_nodes))
        g = self.act(self.l2(g, edge_index, num_nodes))
        fused = F.relu(self.fc_fuse(torch.cat([x, g], dim=-1)))
        fused = self.dropout(fused)
        return self.fc_out(fused)


class TextOnlyClassifier(nn.Module):
    def __init__(self, emb_dim, hidden=256, num_classes=2, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(emb_dim, hidden)
        self.fc2 = nn.Linear(hidden, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index=None, num_nodes=None):
        h = F.relu(self.fc1(x))
        h = self.dropout(h)
        return self.fc2(h)


def resample_train_mask(labels, orig_train_mask, target_fake_ratio, seed):
    rng = np.random.RandomState(seed)
    train_idx = torch.nonzero(orig_train_mask, as_tuple=True)[0].numpy()
    train_labels = labels[orig_train_mask].numpy()
    fake_idx = train_idx[train_labels == 1]
    real_idx = train_idx[train_labels == 0]
    Fk, R = len(fake_idx), len(real_idx)
    p = target_fake_ratio
    new_fake_a = Fk
    new_real_a = round(Fk * (1 - p) / p) if p > 0 else R
    new_real_b = R
    new_fake_b = round(R * p / (1 - p)) if p < 1 else Fk
    if new_real_a <= R:
        new_fake, new_real = new_fake_a, new_real_a
    else:
        new_fake, new_real = new_fake_b, new_real_b
    new_fake = max(1, min(new_fake, Fk))
    new_real = max(1, min(new_real, R))
    chosen_fake = rng.choice(fake_idx, size=new_fake, replace=False)
    chosen_real = rng.choice(real_idx, size=new_real, replace=False)
    chosen = np.concatenate([chosen_fake, chosen_real])
    new_mask = torch.zeros_like(orig_train_mask)
    new_mask[chosen] = True
    return new_mask


def lr_lambda(epoch, total_epochs, warmup_frac=0.1):
    warmup_epochs = max(1, int(total_epochs * warmup_frac))
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return 0.5 * (1 + np.cos(np.pi * progress))


def train_one_run(model_type, src_name, all_data, device, target_fake_ratio, seed,
                   epochs=30, lr=1e-3, weight_decay=1e-4, hidden=256, dropout=0.3,
                   grad_clip_norm=1.0, collapse_threshold=0.98):
    """UNWEIGHTED throughout -- this experiment specifically studies collapse direction,
    which only exists in the unweighted (collapse-prone) regime."""
    set_seed(seed)
    data_src = all_data[src_name]
    x_src = data_src["emb"].to(device)
    y_src = data_src["labels"].to(device)
    val_mask = data_src["val_mask"].to(device)
    test_mask = data_src["test_mask"].to(device)
    edge_index_src = data_src["edge_index"].to(device)
    num_nodes_src = data_src["num_nodes"]

    train_mask = resample_train_mask(data_src["labels"], data_src["train_mask"], target_fake_ratio, seed).to(device)

    emb_dim = x_src.size(1)
    if model_type == "gcn":
        model = HybridGraphTransformer(GCNLayer, emb_dim, hidden, hidden, 2, dropout).to(device)
    elif model_type == "gat":
        model = HybridGraphTransformer(GATLayer, emb_dim, hidden, hidden, 2, dropout).to(device)
    elif model_type == "tbase":
        model = TextOnlyClassifier(emb_dim, hidden, 2, dropout).to(device)
    else:
        raise ValueError(model_type)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda ep: lr_lambda(ep, epochs))
    criterion = nn.CrossEntropyLoss()

    best_val_f1 = -1.0
    best_state = None
    val_f1_curve = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(x_src, edge_index_src, num_nodes_src)
        loss = criterion(logits[train_mask], y_src[train_mask])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(x_src, edge_index_src, num_nodes_src)[val_mask]
            val_pred = val_logits.argmax(dim=-1)
            val_f1 = f1_score(y_src[val_mask].cpu().numpy(), val_pred.cpu().numpy(), average="macro")
        val_f1_curve.append(val_f1)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    model.eval()
    with torch.no_grad():
        test_logits = model(x_src, edge_index_src, num_nodes_src)[test_mask]
        test_pred = test_logits.argmax(dim=-1)
        test_labels = y_src[test_mask]
        test_acc = accuracy_score(test_labels.cpu().numpy(), test_pred.cpu().numpy())
        test_f1 = f1_score(test_labels.cpu().numpy(), test_pred.cpu().numpy(), average="macro")
        test_pred_fake_frac = test_pred.float().mean().item()  # label 1 = fake

    collapsed = (test_pred_fake_frac >= collapse_threshold) or (test_pred_fake_frac <= (1 - collapse_threshold))
    best_checkpoint_epoch = int(np.argmax(val_f1_curve))

    return {
        "model_type": model_type, "source": src_name, "target_fake_ratio": target_fake_ratio,
        "seed": seed, "test_macro_f1": test_f1, "test_acc": test_acc,
        "test_pred_fake_frac": test_pred_fake_frac, "collapsed": bool(collapsed),
        "best_val_macro_f1": best_val_f1, "best_checkpoint_epoch": best_checkpoint_epoch,
    }


def run_extended_seed_sweep(
    base_dir, sources=["coaid", "fakenewsnet"], ratios=[0.5, 0.6, 0.7, 0.8, 0.85],
    seeds=list(range(10)), model_types=["tbase", "gcn", "gat"],
    model_name="roberta-base", epochs=30, max_length=128, batch_size=32, k=10, dropout=0.3,
):
    device = get_device()
    all_data = prepare_all_datasets(base_dir, model_name, max_length, batch_size, k, device, sources)

    out_dir = os.path.join(base_dir, "extended_seed_results")
    os.makedirs(out_dir, exist_ok=True)

    rows = []
    total = len(sources) * len(ratios) * len(seeds) * len(model_types)
    counter = 0
    print(f"\n=== Extended-seed sweep: {total} runs (unweighted, n_seeds={len(seeds)}) ===")
    for model_type in model_types:
        for src in sources:
            for ratio in ratios:
                for seed in seeds:
                    counter += 1
                    result = train_one_run(model_type, src, all_data, device, ratio, seed,
                                            epochs=epochs, dropout=dropout)
                    if counter % 20 == 0 or counter == total:
                        print(f"[{counter}/{total}] {model_type} {src} ratio={ratio} seed={seed} "
                              f"-> F1={result['test_macro_f1']:.4f} collapsed={result['collapsed']}")
                    rows.append(result)

    df = pd.DataFrame(rows)
    out_path = os.path.join(out_dir, "extended_seed_summary.csv")
    df.to_csv(out_path, index=False)
    print(f"\nSaved to: {out_path}")

    # ---- Collapse-direction analysis on collapsed runs only, at n=10 seeds ----
    val_majority = {}
    for src in sources:
        src_df = load_clean_dataset(base_dir, src)
        val_df = src_df[src_df["split"] == "val"]
        real_frac = (val_df["label"] == 0).mean()  # label 0 = real
        val_majority[src] = "real" if real_frac >= 0.5 else "fake"
        print(f"{src}: val real fraction = {real_frac:.4f} (majority = {val_majority[src]})")

    collapsed = df[df["collapsed"]].copy()
    collapsed["direction"] = collapsed["test_pred_fake_frac"].apply(lambda f: "fake" if f >= 0.5 else "real")
    collapsed["matches_val_majority"] = collapsed.apply(
        lambda r: r["direction"] == val_majority[r["source"]], axis=1
    )

    print(f"\n=== Collapse-direction summary at n={len(seeds)} seeds ({len(collapsed)} collapsed runs) ===")
    direction_summary = collapsed.groupby(["model_type", "source", "target_fake_ratio", "direction"]).size().unstack(fill_value=0)
    print(direction_summary)

    overall_rate = collapsed.groupby("model_type")["matches_val_majority"].agg(["mean", "count"])
    print(f"\n=== Fraction of collapsed runs matching true (val) population majority, by architecture ===")
    print(overall_rate)

    dir_path = os.path.join(out_dir, "extended_collapse_direction.csv")
    collapsed.to_csv(dir_path, index=False)
    print(f"\nSaved collapse-direction detail to: {dir_path}")

    return df, collapsed


run_extended_seed_sweep(
    base_dir="drive/MyDrive/CSML/data",
    sources=["coaid", "fakenewsnet"],
    ratios=[0.5, 0.6, 0.7, 0.8, 0.85],
    seeds=list(range(10)),
    model_types=["tbase", "gcn", "gat"],
    model_name="roberta-base",
    epochs=30,
    max_length=128,
    batch_size=32,
    k=10,
    dropout=0.3,
)
